# 🏥 Emergency Department Wait Time & Overcrowding Analysis

## Project Overview
This project analyzes Emergency Department visit data to identify peak
overcrowding periods, high-risk departments, and patient flow bottlenecks
to help hospital operations teams staff smarter and reduce dangerous delays.

## Stakeholder Questions
1. What hours and days of the week have the longest average wait times?
2. Which ED departments are most overcrowded?
3. Does patient acuity level affect how long patients wait?
4. What percentage of patients leave without being seen — and when?
5. Which combination of day, hour, and department creates the worst bottlenecks?

## Tools Used
- Python (pandas, numpy) — data generation and manipulation
- SQLite — SQL analysis (multiple JOINs, subqueries, datetime functions)
- Tableau — heatmap and operational dashboard

## Data
Synthetic datasets generated to simulate real Emergency Department records.

In [9]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# ── DATASET 1: PATIENTS ──────────────────────────────────────────────────────
num_patients = 600
zip_codes = ['10001','10002','10003','10004','10005',
             '10006','10007','10008','10009','10010']

patients_df = pd.DataFrame({
    'patient_id'    : range(1, num_patients + 1),
    'age'           : np.random.randint(1, 95, num_patients),
    'gender'        : np.random.choice(['M','F'], num_patients),
    'zip_code'      : np.random.choice(zip_codes, num_patients),
    'insurance_type': np.random.choice(
                        ['Medicare','Medicaid','Private','Uninsured'],
                        num_patients, p=[0.35, 0.30, 0.25, 0.10])
})

# ── DATASET 2: VISITS ─────────────────────────────────────────────────────────
num_visits = 1000
hour_weights = [1,1,1,1,1,1,2,3,4,4,4,4,4,4,4,5,5,5,5,4,3,2,1,1]
hour_probs   = [w/sum(hour_weights) for w in hour_weights]

arrival_dates = []
for _ in range(num_visits):
    hour = int(np.random.choice(range(24), p=hour_probs))
    arrival = datetime(2024, 1, 1) + timedelta(
        days=random.randint(0, 364),
        hours=hour,
        minutes=random.randint(0, 59))
    arrival_dates.append(arrival)

wait_times = []
for arr in arrival_dates:
    if 14 <= arr.hour <= 22:
        wait = np.random.randint(45, 240)
    elif 7 <= arr.hour <= 13:
        wait = np.random.randint(20, 120)
    else:
        wait = np.random.randint(10, 60)
    wait_times.append(wait)

departure_times = [arr + timedelta(minutes=int(wt))
                   for arr, wt in zip(arrival_dates, wait_times)]

visits_df = pd.DataFrame({
    'visit_id'          : range(1, num_visits + 1),
    'patient_id'        : np.random.randint(1, num_patients + 1, num_visits),
    'arrival_datetime'  : arrival_dates,
    'departure_datetime': departure_times,
    'wait_time_minutes' : wait_times,
    'disposition'       : np.random.choice(
                            ['Discharged','Admitted','LWBS'],
                            num_visits, p=[0.65, 0.25, 0.10])
})

# ── DATASET 3: TRIAGE ─────────────────────────────────────────────────────────
triage_df = pd.DataFrame({
    'triage_id'      : range(1, num_visits + 1),
    'visit_id'       : range(1, num_visits + 1),
    'department'     : np.random.choice(
                         ['Trauma','Cardiac','Pediatrics','General','Psychiatric'],
                         num_visits, p=[0.15, 0.20, 0.15, 0.35, 0.15]),
    'acuity_level'   : np.random.choice([1,2,3,4,5], num_visits,
                         p=[0.05, 0.15, 0.40, 0.30, 0.10]),
    'chief_complaint': np.random.choice([
                         'Chest Pain','Shortness of Breath','Abdominal Pain',
                         'Fever','Laceration','Fracture','Altered Mental Status',
                         'Headache', 'Dizziness', 'Nausea/Vomiting', 'Back Pain',
                         'Weakness', 'Syncope', 'Seizure'],
                         num_visits)
})
print("✅ Patients dataset:", patients_df.shape)
print("✅ Visits dataset:", visits_df.shape)
print("✅ Triage dataset:", triage_df.shape)
print("\n--- Patients Sample ---")
print(patients_df.head())
print("\n--- Visits Sample ---")
print(visits_df.head())
print("\n--- Triage Sample ---")
print(triage_df.head())

✅ Patients dataset: (600, 5)
✅ Visits dataset: (1000, 6)
✅ Triage dataset: (1000, 5)

--- Patients Sample ---
   patient_id  age gender zip_code insurance_type
0           1   52      F    10008       Medicaid
1           2   93      M    10009       Medicare
2           3   15      F    10005        Private
3           4   72      M    10003      Uninsured
4           5   61      M    10005      Uninsured

--- Visits Sample ---
   visit_id  patient_id    arrival_datetime  departure_datetime  \
0         1         540 2024-11-23 16:07:00 2024-11-23 19:10:00   
1         2         191 2024-01-13 02:47:00 2024-01-13 03:33:00   
2         3         315 2024-05-20 05:15:00 2024-05-20 05:49:00   
3         4         563 2024-04-24 17:08:00 2024-04-24 18:10:00   
4         5         562 2024-02-22 05:43:00 2024-02-22 06:11:00   

   wait_time_minutes disposition  
0                183  Discharged  
1                 46        LWBS  
2                 34  Discharged  
3                 62    

In [10]:
# Create SQLite database in memory
conn = sqlite3.connect(':memory:')

# Load all three dataframes as SQL tables
patients_df.to_sql('patients', conn, index=False, if_exists='replace')
visits_df.to_sql('visits', conn, index=False, if_exists='replace')
triage_df.to_sql('triage', conn, index=False, if_exists='replace')

print("✅ All 3 tables loaded into SQLite successfully!")
print("\nTables available:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

✅ All 3 tables loaded into SQLite successfully!

Tables available:
       name
0  patients
1    visits
2    triage


## Query 1 — Average Wait Time by Hour and Day of Week
**Stakeholder Question 1:** What hours and days of the week have the longest
average wait times?

Evening hours (8pm–10pm) show the longest waits averaging 175–225 minutes.
Friday evenings peak at 225 minutes. Evening overcrowding appears systemic
across all days of the week — not isolated to weekends.

In [11]:
query1 = """
SELECT
    CAST(strftime('%H', arrival_datetime) AS INTEGER)  AS hour,
    strftime('%w', arrival_datetime)                   AS day_of_week,
    ROUND(AVG(wait_time_minutes), 1)                   AS avg_wait_time
FROM visits v
GROUP BY hour, day_of_week
ORDER BY hour ASC, day_of_week ASC;
"""

wait_by_hour_day = pd.read_sql(query1, conn)
print("Total rows:", len(wait_by_hour_day))
print(wait_by_hour_day.head(10))

Total rows: 159
   hour day_of_week  avg_wait_time
0     0           0           58.0
1     0           1           35.0
2     0           2           31.8
3     0           3           29.5
4     0           4           40.5
5     0           6           41.4
6     1           0           35.3
7     1           1           33.0
8     1           2           36.5
9     1           5           54.0


## Query 2 — Most Overcrowded ED Departments
**Stakeholder Question 2:** Which ED departments are most overcrowded?

General has the highest patient volume (331 visits) while Cardiac has the
longest average wait at 105 minutes. Every department exceeds 100 minutes
average wait time — indicating systemic overcrowding rather than an isolated
departmental problem.

In [12]:
query2 = """
SELECT
    t.department,
    COUNT(*)                    AS total_visits,
    ROUND(AVG(v.wait_time_minutes), 1)  AS avg_wait_time
FROM triage t
JOIN visits v ON t.visit_id = v.visit_id
GROUP BY t.department
ORDER BY total_visits DESC;
"""

dept_overcrowding = pd.read_sql(query2, conn)
print(dept_overcrowding)

    department  total_visits  avg_wait_time
0      General           331          102.6
1      Cardiac           201          105.0
2       Trauma           179          104.5
3  Psychiatric           153          100.3
4   Pediatrics           136          101.9


## Query 3 — Wait Time by Acuity Level
**Stakeholder Question 3:** Does patient acuity level affect how long
patients wait?

Critical finding: All acuity levels wait approximately 100 minutes regardless
of severity. Level 1 (life-threatening) patients wait 100.8 minutes vs Level 5
(non-urgent) at 101 minutes — a difference of less than one minute. The triage
system is not functioning as designed. This is a patient safety issue requiring
immediate review by medical leadership.

In [13]:
query3 = """
SELECT
    t.acuity_level,
    COUNT(*)                            AS total_patients,
    ROUND(AVG(v.wait_time_minutes), 1)  AS avg_wait_time
FROM triage t
JOIN visits v ON t.visit_id = v.visit_id
GROUP BY t.acuity_level
ORDER BY t.acuity_level ASC;
"""

acuity_wait = pd.read_sql(query3, conn)
print(acuity_wait)

   acuity_level  total_patients  avg_wait_time
0             1              48          100.8
1             2             137          101.5
2             3             399          106.3
3             4             326          100.3
4             5              90          101.0


## Query 4 — LWBS Rate by Hour
**Stakeholder Question 4:** What percentage of patients leave without
being seen and when?

8am has the highest LWBS rate at 19.1% — likely tied to morning shift
changes. No consistent time-of-day pattern exists, suggesting LWBS is
driven by sudden volume spikes rather than predictable daily cycles. Based on healthcare operations experience. My next step would be to validate that by analyzing wait time or staffing data.

In [14]:
query4 = """
SELECT
    strftime('%H', v.arrival_datetime)     AS hour,
    COUNT(*)                               AS total_visits,
    SUM(CASE WHEN v.disposition = 'LWBS'
             THEN 1 ELSE 0 END)            AS lwbs_count,
    ROUND(SUM(CASE WHEN v.disposition = 'LWBS'
             THEN 1 ELSE 0 END) * 100.0
             / COUNT(*), 1)               AS lwbs_rate_pct
FROM visits v
GROUP BY hour
ORDER BY lwbs_rate_pct DESC
LIMIT 10;
"""

lwbs_by_hour = pd.read_sql(query4, conn)
print(lwbs_by_hour)

  hour  total_visits  lwbs_count  lwbs_rate_pct
0   08            47           9           19.1
1   02            11           2           18.2
2   13            61           8           13.1
3   22            18           2           11.1
4   00            18           2           11.1
5   19            56           6           10.7
6   23            10           1           10.0
7   16            74           7            9.5
8   10            66           6            9.1
9   09            45           4            8.9


## Query 5 — Worst Bottleneck Combinations
**Stakeholder Question 5:** Which combination of day, hour, and department
creates the worst bottlenecks?

Initial analysis suggests Friday afternoons and evenings may experience longer ED wait times, particularly in Trauma, Cardiac, and Pediatrics. However, most of these time-slot combinations are based on very small sample sizes (1–2 visits), so these findings should be treated as directional rather than conclusive. Additional data over a longer time period would be needed before making staffing adjustments.

In [15]:
query5 = """
SELECT
    strftime('%H', v.arrival_datetime)  AS hour,
    strftime('%w', v.arrival_datetime)  AS day_of_week,
    t.department,
    COUNT(*)                            AS total_visits,
    ROUND(AVG(v.wait_time_minutes), 1)  AS avg_wait_time
FROM visits v
JOIN triage t ON v.visit_id = t.visit_id
GROUP BY hour, day_of_week, t.department
ORDER BY avg_wait_time DESC
LIMIT 10;
"""

worst_bottlenecks = pd.read_sql(query5, conn)
print(worst_bottlenecks)

  hour day_of_week   department  total_visits  avg_wait_time
0   21           5   Pediatrics             1          238.0
1   14           3       Trauma             1          236.0
2   22           1      General             1          234.0
3   17           3       Trauma             1          233.0
4   21           5      Cardiac             1          231.0
5   17           1  Psychiatric             1          230.0
6   14           4      Cardiac             1          228.0
7   16           5       Trauma             1          226.0
8   15           5       Trauma             2          222.5
9   17           1      Cardiac             1          221.0
